# 03 - Kombinierte Multiplot-Tabellen erzeugen

Dieses Notebook verbindet Bulk- und Oberflächen-Ergebnisse zu den Tabellen, die
unter dem Multiplot angezeigt werden. Es startet mit den HDF-Dateien, kann aber
auch die in den vorherigen Notebooks gespeicherten Tutorial-CSV-Dateien
verwenden.

In [ ]:
from pathlib import Path
import re
import sys

import numpy as np
import pandas as pd

# Compatibility shim for HDF files written in a different NumPy/PyTables stack.
# Keep this cell at the top before calling pd.read_hdf.
try:
    import numpy.core as npc
    sys.modules.setdefault("numpy._core", npc)
    sys.modules.setdefault("numpy._core.multiarray", np.core.multiarray)
    sys.modules.setdefault("numpy._core.numeric", np.core.numeric)
except Exception as exc:
    print("NumPy compatibility shim skipped:", exc)

DATA_ROOT = Path(r"/Users/dk2994/Desktop/Uni/Journal/Thesis/Notebooks/Surface Alloys")
PROJECT_ROOT = Path(r"/Users/dk2994/Desktop/git/PFUI")
OUTPUT_ROOT = Path(r"/Users/dk2994/Desktop/git/PFUI/notebooks/phase_diagram_outputs")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

HDF_FILES = {
    "bulk": DATA_ROOT / "CuGabulk.hdf",
    "bulk_oxide": DATA_ROOT / "CuGabulk_oxide.hdf",
    "surface_100": DATA_ROOT / "CuGasurf_100.hdf",
    "surface_110": DATA_ROOT / "CuGasurf_110.hdf",
    "surface_111": DATA_ROOT / "CuGasurf_111.hdf",
    "surface_211": DATA_ROOT / "CuGasurf_211.hdf",
}

def read_onepiece_hdf(path, key="df"):
    """Read a OnePiece-exported pandas HDF table.

    OnePiece stores simulation records in a pandas DataFrame. The HDF files in
    this tutorial are read with pd.read_hdf(filename, key="df").
    """
    path = Path(path)
    frame = pd.read_hdf(path, key=key)
    frame.attrs["source_hdf"] = str(path)
    return frame

def formula_counts(formula):
    if not isinstance(formula, str):
        return {}
    counts = {}
    for element, number in re.findall(r"([A-Z][a-z]?)(\d*)", formula):
        counts[element] = counts.get(element, 0) + int(number or 1)
    return counts

## Vorhandene Zwischenergebnisse laden

In [ ]:
bulk_summary_path = OUTPUT_ROOT / "tutorial_bulk_transition_summary.csv"
surface_summary_path = OUTPUT_ROOT / "tutorial_surface_all_stable_phases.csv"

if not bulk_summary_path.exists() or not surface_summary_path.exists():
    raise FileNotFoundError(
        "Run notebooks 01 and 02 first, or copy their CSV outputs into phase_diagram_outputs."
    )

bulk_summary = pd.read_csv(bulk_summary_path)
surface_all = pd.read_csv(surface_summary_path)
bulk_summary.head(), surface_all.head()

## Spalten auf ein gemeinsames Schema bringen

In [ ]:
bulk_table = bulk_summary.copy()
bulk_table["surface_or_bulk"] = "bulk"
bulk_table["hkl"] = ""
bulk_table["Monolayer_alloy"] = np.nan
bulk_table["Cu_atoms"] = bulk_table.get("Cu_atoms", np.nan)
bulk_table["Ga_atoms"] = bulk_table.get("Ga_atoms", np.nan)
bulk_table["energy_column"] = "formation_energy_per_atom"

surface_table = surface_all.rename(columns={
    "T_min_stable_K": "T_min_K",
    "T_max_stable_K": "T_max_K",
    "log10_ratio_min_stable": "log10_ratio_min",
    "log10_ratio_max_stable": "log10_ratio_max",
    "min_G_per_Area_eV_A2": "min_energy",
})
surface_table["panel"] = "Surface hkl " + surface_table["hkl"].astype(str)
surface_table["surface_or_bulk"] = np.where(
    surface_table["Name"].str.contains("clean", case=False, na=False),
    "clean surface",
    "Ga-covered surface",
)
surface_table["Ga_percent"] = np.nan
surface_table["Cu_atoms"] = surface_table["Cu"]
surface_table["Ga_atoms"] = surface_table["Ga"]
surface_table["unit"] = "eV/Å²"
surface_table["energy_column"] = "G_per_Area_corrected"

columns = [
    "panel", "surface_or_bulk", "hkl", "phase_id", "Name", "Formula",
    "phase_label", "Ga_percent", "Monolayer_alloy", "Cu_atoms", "Ga_atoms",
    "stable_grid_fraction", "stable_percent", "T_min_K", "T_max_K",
    "log10_ratio_min", "log10_ratio_max", "min_energy", "unit", "energy_column",
]

combined = pd.concat([bulk_table[columns], surface_table[columns]], ignore_index=True)
combined["T_span_K"] = combined["T_max_K"] - combined["T_min_K"]
combined["log10_ratio_span"] = combined["log10_ratio_max"] - combined["log10_ratio_min"]
combined = combined.sort_values(["panel", "stable_grid_fraction"], ascending=[True, False])
combined.insert(0, "rank_in_panel", combined.groupby("panel").cumcount() + 1)
combined.head(12)

In [ ]:
combined.to_csv(OUTPUT_ROOT / "tutorial_bulk_surface_transition_phase_summary_extended.csv", index=False)
combined

## Multiplot-Tabelle wie im HTML-Output

In [ ]:
for panel, table in combined.groupby("panel"):
    display_cols = [
        "rank_in_panel", "surface_or_bulk", "hkl", "Name", "Formula",
        "phase_label", "Monolayer_alloy", "Cu_atoms", "Ga_atoms",
        "stable_percent", "T_min_K", "T_max_K",
        "log10_ratio_min", "log10_ratio_max", "min_energy", "unit",
    ]
    print("\n" + panel)
    display(table[display_cols])

## Verbindung zum vorhandenen Multiplot

Der vorhandene HTML-Multiplot verwendet dieselbe Idee:

1. Pro Rasterpunkt werden Energien für alle Kandidaten berechnet.
2. `argmin` bestimmt die stabile Phase.
3. Eine Summary-Tabelle zählt, wie oft jede Phase stabil ist.
4. Die Tabellen werden pro Panel gruppiert und als HTML ausgegeben.

Wenn du die bereits erzeugten finalen Dateien erweitern willst, nutze:
`notebooks/build_transition_multiplot_tables.py`.

In [ ]:
final_existing = OUTPUT_ROOT / "cuga_bulk_surface_transition_phase_summary_extended.csv"
if final_existing.exists():
    final_table = pd.read_csv(final_existing)
    print(final_existing)
    display(final_table.head())
else:
    print("Final extended table not found yet.")